# PosEnc Output Comparison

This notebook compares encoder behavior from saved artifacts and probe-based diagnostics.

- Positional heatmaps are computed in **probe mode** (not from random saved vectors).
- Attention plots use **independent synthetic q/k vectors** (rope-graph style).
- Heatmaps use shared scales across encoders for direct visual comparability.
            


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print('matplotlib unavailable:', exc)
    print('Install notebook deps with: uv sync --extra notebooks')
            


In [ ]:
REPO_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/home/jake/Developer/posenc'),
]

REPO_ROOT = next(
    (
        p.resolve()
        for p in REPO_CANDIDATES
        if (p / 'core' / 'types.py').exists() and (p / 'encoders' / '__init__.py').exists()
    ),
    Path('/home/jake/Developer/posenc').resolve(),
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('REPO_ROOT:', REPO_ROOT)

CANDIDATE_RUN_DIRS = [
    REPO_ROOT / 'out',
    Path('out').resolve(),
    Path('../out').resolve(),
]

RUN_DIR = next((p for p in CANDIDATE_RUN_DIRS if p.exists()), CANDIDATE_RUN_DIRS[0])
print('RUN_DIR:', RUN_DIR)


def load_artifacts(run_dir: Path):
    meta_path = run_dir / 'metadata.json'
    vectors_path = run_dir / 'vectors.npy'
    if not meta_path.exists() or not vectors_path.exists():
        raise FileNotFoundError(
            f'Missing artifacts in {run_dir}. Expected metadata.json and vectors.npy.'
        )

    metadata = json.loads(meta_path.read_text())
    vectors = np.load(vectors_path)
    encoded = {
        p.stem.replace('encoded_', ''): np.load(p)
        for p in sorted(run_dir.glob('encoded_*.npy'))
    }
    return metadata, vectors, encoded


try:
    metadata, vectors, encoded = load_artifacts(RUN_DIR)
    print('encoders with saved tensors:', list(encoded.keys()))
    print('vectors shape:', vectors.shape)
except FileNotFoundError as exc:
    print(exc)
    print('Generate artifacts with:')
    print('uv run python main.py --encoders all --save-dir out --save-encoded')
    metadata = None
    vectors = None
    encoded = {}
            


In [ ]:
from core.positions import build_position_bank, parse_coords, parse_t_values
from core.types import RunConfig
from encoders import resolve_specs

cfg = None
bank = None
specs = []
spec_by_name = {}
caches = {}

if metadata is None:
    print('No metadata loaded; skipping encoder-driven analyses.')
else:
    cfg_meta = metadata.get('config', {})

    raw_coords = cfg_meta.get('coords', ['x', 'y'])
    if isinstance(raw_coords, list):
        coords_text = ','.join(str(x) for x in raw_coords)
    else:
        coords_text = str(raw_coords)

    raw_t_values = cfg_meta.get('t_values', [0.0])
    t_values = parse_t_values(raw_t_values)

    raw_encoder_params = cfg_meta.get('encoder_params', {})
    encoder_params = raw_encoder_params if isinstance(raw_encoder_params, dict) else {}

    encoder_names = metadata.get('encoders', list(encoded.keys()))
    if not encoder_names:
        encoder_names = list(encoded.keys())

    dim = int(cfg_meta.get('dim', vectors.shape[1] if vectors is not None else 64))
    num_vectors_cfg = int(cfg_meta.get('num_vectors', 2))

    cfg = RunConfig(
        encoders=tuple(str(x) for x in encoder_names),
        dim=dim,
        num_vectors=max(2, num_vectors_cfg),
        seed=int(cfg_meta.get('seed', 0)),
        theta_base=float(cfg_meta.get('theta_base', 10000.0)),
        coords_spec=parse_coords(coords_text),
        grid_size=int(cfg_meta.get('grid_size', 16)),
        centered_coords=bool(cfg_meta.get('centered_coords', False)),
        t_values=t_values,
        z_value=float(cfg_meta.get('z_value', 0.0)),
        position_chunk_size=int(cfg_meta.get('position_chunk_size', 128)),
        save_dir=None,
        save_encoded=False,
        encoder_params=encoder_params,
    )

    bank = build_position_bank(
        cfg.coords_spec,
        cfg.grid_size,
        cfg.centered_coords,
        cfg.t_values,
        cfg.z_value,
    )
    specs = resolve_specs(list(cfg.encoders))
    spec_by_name = {spec.name: spec for spec in specs}
    caches = {name: spec_by_name[name].precompute(cfg, bank) for name in spec_by_name}

    print('analysis encoders:', list(spec_by_name.keys()))
    print('position count (rope):', int(bank.rope_positions.shape[0]))
            


In [ ]:
if not encoded:
    print('No encoded tensors loaded; skipping saved-output norm summary.')
else:
    mean_norm_by_encoder = {}
    for name, tensor in encoded.items():
        norms = np.linalg.norm(tensor, axis=2)
        mean_norm_by_encoder[name] = float(np.mean(norms))

    if HAVE_MPL:
        plt.figure(figsize=(8, 4))
        plt.bar(mean_norm_by_encoder.keys(), mean_norm_by_encoder.values())
        plt.ylabel('mean output norm')
        plt.title('Mean encoded norm by encoder (saved outputs)')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Mean encoded norm by encoder:')
        for name, value in mean_norm_by_encoder.items():
            print(f'  {name}: {value:.6f}')
            


In [ ]:
if not spec_by_name:
    print('No encoder specs/caches available; skipping probe positional heatmaps.')
elif not HAVE_MPL:
    print('matplotlib unavailable; skipping probe positional heatmaps.')
else:
    probe_idx = 0
    max_positions = 64
    max_dims = 64

    probe = np.zeros((1, cfg.dim), dtype=np.float64)
    probe[:, 0::2] = 1.0

    pair_align_names = {'rope', 'axial', 'spiral', 'ape'}
    mats = {}

    for name in cfg.encoders:
        if name not in spec_by_name:
            continue
        spec = spec_by_name[name]
        out = spec.apply(probe, caches[name], cfg.position_chunk_size)
        mat = out[0]

        if name in pair_align_names:
            aligned = np.empty_like(mat)
            aligned[:, 0::2] = mat[:, 1::2]
            aligned[:, 1::2] = mat[:, 0::2]
            mat = aligned

        p_lim = min(max_positions, mat.shape[0])
        d_lim = min(max_dims, mat.shape[1])
        mats[name] = (mat[:p_lim, :d_lim], p_lim, d_lim)

    if not mats:
        print('No probe heatmaps built.')
    else:
        global_min = min(float(np.min(view)) for view, _, _ in mats.values())
        global_max = max(float(np.max(view)) for view, _, _ in mats.values())
        if abs(global_max - global_min) < 1e-12:
            global_max = global_min + 1e-12

        print(f'Probe positional heatmap shared range: [{global_min:.6f}, {global_max:.6f}]')
        print(f'Display window: positions<= {max_positions}, dimensions<= {max_dims}')

        for name in cfg.encoders:
            if name not in mats:
                continue
            view, p_lim, d_lim = mats[name]
            fig, ax = plt.subplots(figsize=(12, 4.8), constrained_layout=True)
            im = ax.pcolormesh(view, cmap='viridis', shading='auto', vmin=global_min, vmax=global_max)
            ax.set_title(f'{name}: Probe Position vs Encoding Value')
            ax.set_xlabel('Embedding dimension')
            ax.set_ylabel('Token position')
            ax.set_xlim((0, d_lim))
            ax.set_ylim((p_lim, 0))
            cbar = fig.colorbar(im, ax=ax, pad=0.02)
            cbar.set_label('encoding value')
            plt.show()
            


In [ ]:
def offset_curve(sim: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    seq_len = sim.shape[0]
    offsets = np.arange(-(seq_len - 1), seq_len)
    means = np.array([np.mean(np.diag(sim, k=o)) for o in offsets], dtype=np.float64)
    return offsets, means

similarities = {}
offset_curves = {}

if not spec_by_name:
    print('No encoder specs/caches available; skipping synthetic q/k attention analysis.')
else:
    rng = np.random.default_rng(0)
    q0 = rng.normal(size=(cfg.dim,)).astype(np.float64)
    k0 = rng.normal(size=(cfg.dim,)).astype(np.float64)
    base = np.stack((q0, k0), axis=0)

    for name in cfg.encoders:
        if name not in spec_by_name:
            continue
        spec = spec_by_name[name]
        out = spec.apply(base, caches[name], cfg.position_chunk_size)
        q = out[0]
        k = out[1]

        # rope_graph-style: unscaled dot product with independent q/k probes.
        sim = q @ k.T
        similarities[name] = sim
        offset_curves[name] = offset_curve(sim)

    print('Built synthetic q/k attention analyses for encoders:', list(offset_curves.keys()))
            


In [ ]:
if not similarities:
    print('No similarity matrices to plot.')
elif not HAVE_MPL:
    print('matplotlib unavailable; skipping similarity heatmaps.')
else:
    global_abs_max = max(float(np.max(np.abs(sim))) for sim in similarities.values())
    if global_abs_max < 1e-12:
        global_abs_max = 1e-12

    print(f'Similarity heatmap shared symmetric range: [-{global_abs_max:.6f}, +{global_abs_max:.6f}]')

    for name in cfg.encoders:
        if name not in similarities:
            continue
        sim = similarities[name]
        fig, ax = plt.subplots(figsize=(7.5, 6.0), constrained_layout=True)
        im = ax.imshow(sim, cmap='coolwarm', aspect='auto', vmin=-global_abs_max, vmax=global_abs_max)
        ax.set_title(f'{name}: Attention Similarity Matrix (synthetic q/k)')
        ax.set_xlabel('q position')
        ax.set_ylabel('p position')
        cbar = fig.colorbar(im, ax=ax, pad=0.02)
        cbar.set_label('attention similarity')
        plt.show()
            


In [ ]:
if not offset_curves:
    print('No offset curves to plot.')
elif not HAVE_MPL:
    print('Offset-curve summary (first 5 points per encoder):')
    for name, (offsets, values) in offset_curves.items():
        head = ', '.join(f'{int(o)}:{v:.3f}' for o, v in zip(offsets[:5], values[:5]))
        print(f'  {name}: {head}')
else:
    plt.figure(figsize=(10, 5))
    for name in cfg.encoders:
        if name not in offset_curves:
            continue
        offsets, values = offset_curves[name]
        plt.plot(offsets, values, label=name, linewidth=2)

    plt.axvline(0, color='black', linewidth=1, alpha=0.3)
    plt.title('Attention Score vs Positional Offset (synthetic q/k)')
    plt.xlabel('positional offset (p - q)')
    plt.ylabel('mean dot product')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
            
